In [1]:
from app.snowflake_engine import query


In [2]:
def get_dock_data():
    statement = """
    select r.name as region_name, cargo, cruise, other, type, dr.year
    from dock_revenue dr 
    join dock_type dt on dr.dock_type_id = dt.dock_type_id
    join region r on dr.region_id = r.region_id;
    """
    return query(statement)

In [4]:
dock_revenue = get_dock_data()
dock_revenue

/home/flx_franklxu/Fish_Population/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


,REGION_NAME,CARGO,CRUISE,OTHER,TYPE,YEAR
0,Tampa Bay Region,5813,1032.0,659,Dockage,2016
1,Tampa Bay Region,6926,1203.0,229,Dockage,2017
2,Tampa Bay Region,7402,1277.0,463,Dockage,2018
3,Tampa Bay Region,7270,1340.0,845,Dockage,2019
4,Tampa Bay Region,7313,668.0,733,Dockage,2020
5,Tampa Bay Region,8460,NaN,751,Dockage,2021
6,Tampa Bay Region,10262,630.0,1213,Dockage,2022
7,Tampa Bay Region,11446,1144.0,2279,Dockage,2023
8,Tampa Bay Region,11825,1173.0,1394,Dockage,2024
9,Tampa Bay Region,11234,1488.0,1742,Dockage,2025


In [9]:
out = (
    dock_revenue.pivot_table(
        index='YEAR',
        columns='TYPE',
        values=['CARGO','CRUISE','OTHER'],
        aggfunc='sum',
    )
    .reset_index()
)
out

YEAR   CARGO           CRUISE            OTHER         
TYPE       Dockage Wharfage Dockage Wharfage Dockage Wharfage
0     2016    5813     9791  1032.0   6371.0     659     8706
1     2017    6926    10561  1203.0   8041.0     229     8231
2     2018    7402    11991  1277.0   8868.0     463     8356
3     2019    7270    11521  1340.0  10178.0     845     9942
4     2020    7313    11892   668.0   4735.0     733     8767
5     2021    8460    13026     0.0      0.0     751     9747
6     2022   10262    12794   630.0   4411.0    1213    11274
7     2023   11446    14266  1144.0  10791.0    2279    10853
8     2024   11825    14426  1173.0  12066.0    1394    10161
9     2025   11234    13734  1488.0  17744.0    1742    12275

In [21]:
dock_revenue.set_index(['YEAR','TYPE'])[['CARGO','CRUISE','OTHER']]

,,CARGO,CRUISE,OTHER
YEAR,TYPE,,,
2016,Dockage,5813,1032.0,659
2017,Dockage,6926,1203.0,229
2018,Dockage,7402,1277.0,463
2019,Dockage,7270,1340.0,845
2020,Dockage,7313,668.0,733
2021,Dockage,8460,NaN,751
2022,Dockage,10262,630.0,1213
2023,Dockage,11446,1144.0,2279
2024,Dockage,11825,1173.0,1394


In [22]:
dock_revenue.set_index(['YEAR','TYPE'])[['CARGO','CRUISE','OTHER']].unstack('TYPE')

CARGO           CRUISE            OTHER         
TYPE Dockage Wharfage Dockage Wharfage Dockage Wharfage
YEAR                                                   
2016    5813     9791  1032.0   6371.0     659     8706
2017    6926    10561  1203.0   8041.0     229     8231
2018    7402    11991  1277.0   8868.0     463     8356
2019    7270    11521  1340.0  10178.0     845     9942
2020    7313    11892   668.0   4735.0     733     8767
2021    8460    13026     NaN      NaN     751     9747
2022   10262    12794   630.0   4411.0    1213    11274
2023   11446    14266  1144.0  10791.0    2279    10853
2024   11825    14426  1173.0  12066.0    1394    10161
2025   11234    13734  1488.0  17744.0    1742    12275

In [24]:
out = dock_revenue.set_index(['YEAR','TYPE'])[['CARGO','CRUISE','OTHER']].unstack('TYPE').reset_index()
out

YEAR   CARGO           CRUISE            OTHER         
TYPE       Dockage Wharfage Dockage Wharfage Dockage Wharfage
0     2016    5813     9791  1032.0   6371.0     659     8706
1     2017    6926    10561  1203.0   8041.0     229     8231
2     2018    7402    11991  1277.0   8868.0     463     8356
3     2019    7270    11521  1340.0  10178.0     845     9942
4     2020    7313    11892   668.0   4735.0     733     8767
5     2021    8460    13026     NaN      NaN     751     9747
6     2022   10262    12794   630.0   4411.0    1213    11274
7     2023   11446    14266  1144.0  10791.0    2279    10853
8     2024   11825    14426  1173.0  12066.0    1394    10161
9     2025   11234    13734  1488.0  17744.0    1742    12275

In [25]:
out.columns

MultiIndex([(  'YEAR',         ''),
            ( 'CARGO',  'Dockage'),
            ( 'CARGO', 'Wharfage'),
            ('CRUISE',  'Dockage'),
            ('CRUISE', 'Wharfage'),
            ( 'OTHER',  'Dockage'),
            ( 'OTHER', 'Wharfage')],
           names=[None, 'TYPE'])

In [19]:
out.columns = [f'{m}_{t}' if t else m for m, t in out.columns]

In [20]:
out

,YEAR,CARGO_Dockage,CARGO_Wharfage,CRUISE_Dockage,CRUISE_Wharfage,OTHER_Dockage,OTHER_Wharfage
0,2016,5813,9791,1032.0,6371.0,659,8706
1,2017,6926,10561,1203.0,8041.0,229,8231
2,2018,7402,11991,1277.0,8868.0,463,8356
3,2019,7270,11521,1340.0,10178.0,845,9942
4,2020,7313,11892,668.0,4735.0,733,8767
5,2021,8460,13026,NaN,NaN,751,9747
6,2022,10262,12794,630.0,4411.0,1213,11274
7,2023,11446,14266,1144.0,10791.0,2279,10853
8,2024,11825,14426,1173.0,12066.0,1394,10161
9,2025,11234,13734,1488.0,17744.0,1742,12275


In [26]:
def get_seagrass_data():
    statement = """
    select r.name as region_name, w.name as water_body_name, acres, year
    from seagrass g join water w on g.water_id = w.water_id
    join region r on r.region_id = w.region_id;
    """
    return query(statement)

def get_fish_data():
    statement = """
    select common_name, scientific_name, name as region_name, 
    avg_size, max_size, min_size, sample_size, date
    from fish_data fd join fish f on fd.fish_id = f.fish_id
    join region r on fd.region_id = r.region_id;
    """
    return query(statement)

def get_water_quality_data():
    statement = """
    select r.name as region_name, w.name as water_body_name, nh3_n_mgl, no3_mgl, ph, 
    salinity, water_temp, date
    from water_quality wq join water w on wq.water_id = w.water_id
    join region r on r.region_id = w.region_id;
    """
    return query(statement)

def get_revenue_data():
    statement = """
    select r.name as region_name, dry, general, liquid, passenger_count, 
    revenue, land_lease, net_investment, port_usage_fee, car.year
    from cargo car join cruise cru on car.region_id = cru.region_id
    join region_revenue rr on rr.region_id = car.region_id
    join region r on r.region_id = rr.region_id;
    """
    return query(statement)

def get_dock_data():
    statement = """
    select r.name as region_name, cargo, cruise, other, type, dr.year
    from dock_revenue dr 
    join dock_type dt on dr.dock_type_id = dt.dock_type_id
    join region r on dr.region_id = r.region_id;
    """
    return query(statement)

In [35]:
def yearify_date(data, agg_columns, numeric_columns):
    data = data.set_index('DATE').sort_index()
    yearly_avg = data.groupby(agg_columns).resample('YE').agg({
        col: 'mean' for col in numeric_columns
    }).reset_index()
    yearly_avg['YEAR'] = yearly_avg['DATE'].dt.year
    yearly_avg = yearly_avg.drop('DATE', axis=1)
    return yearly_avg

In [36]:
agg_columns = ['SCIENTIFIC_NAME', 'COMMON_NAME']
numeric_columns = ['SAMPLE_SIZE', 'AVG_SIZE', 'MAX_SIZE', 'MIN_SIZE']
fish_data = get_fish_data()
yearly_fish = yearify_date(fish_data, agg_columns, numeric_columns).dropna()
counts = yearly_fish.groupby(agg_columns)['SCIENTIFIC_NAME'].transform('size')
filtered_fish = yearly_fish[counts > 10]

agg_columns = ['REGION_NAME']
numeric_columns = ['NH3_N_MGL', 'NO3_MGL', 'PH', 'SALINITY', 'WATER_TEMP']
water_data = get_water_quality_data()
yearly_water = yearify_date(water_data, agg_columns, numeric_columns).dropna()

dock_revenue = get_dock_data()
dock_revenue = dock_revenue.set_index(['YEAR','TYPE'])[['CARGO','CRUISE','OTHER']].unstack('TYPE').reset_index()
dock_revenue.columns = [f'{m}_{t}' if t else m for m, t in dock_revenue.columns]

region_revenue = get_revenue_data()

seagrass_data = get_seagrass_data()
yearly_acrage = seagrass_data.groupby('YEAR')[['ACRES']].sum().reset_index()

/home/flx_franklxu/Fish_Population/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(
/home/flx_franklxu/Fish_Population/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserve

In [62]:
dock_revenue

,YEAR,CARGO_Dockage,CARGO_Wharfage,CRUISE_Dockage,CRUISE_Wharfage,OTHER_Dockage,OTHER_Wharfage
0,2016,5813,9791,1032.0,6371.0,659,8706
1,2017,6926,10561,1203.0,8041.0,229,8231
2,2018,7402,11991,1277.0,8868.0,463,8356
3,2019,7270,11521,1340.0,10178.0,845,9942
4,2020,7313,11892,668.0,4735.0,733,8767
5,2021,8460,13026,NaN,NaN,751,9747
6,2022,10262,12794,630.0,4411.0,1213,11274
7,2023,11446,14266,1144.0,10791.0,2279,10853
8,2024,11825,14426,1173.0,12066.0,1394,10161
9,2025,11234,13734,1488.0,17744.0,1742,12275


In [83]:
aggregate_data = (
    filtered_fish
    .merge(yearly_water, how='inner', on='YEAR')
)
aggregate_data

,SCIENTIFIC_NAME,COMMON_NAME,SAMPLE_SIZE,AVG_SIZE,MAX_SIZE,MIN_SIZE,YEAR,REGION_NAME,NH3_N_MGL,NO3_MGL,PH,SALINITY,WATER_TEMP
0,Acanthostracion quadricornis,Scrawled Cowfish,2.000000,109.483146,120.696629,99.887640,1996,Tampa Bay Region,0.003012,0.0,8.014864,29.979476,75.946684
1,Acanthostracion quadricornis,Scrawled Cowfish,2.307018,110.333333,124.043860,98.877193,1997,Tampa Bay Region,0.000000,0.0,7.986507,31.189481,78.286941
2,Acanthostracion quadricornis,Scrawled Cowfish,2.116279,145.627907,161.418605,133.348837,1998,Tampa Bay Region,0.000000,0.0,7.968170,26.685966,77.640922
3,Acanthostracion quadricornis,Scrawled Cowfish,1.895833,127.791667,134.500000,121.604167,1999,Tampa Bay Region,0.000000,0.0,8.000000,30.220868,76.806000
4,Acanthostracion quadricornis,Scrawled Cowfish,2.744186,127.534884,146.488372,112.023256,2000,Tampa Bay Region,0.000000,0.0,8.067199,31.834402,79.288026
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2828,Urophycis floridana,Southern Hake,1.363636,66.090909,68.363636,62.909091,2009,Tampa Bay Region,0.038911,0.0,8.281035,30.639543,97.057990
2829,Urophycis floridana,Southern Hake,2.515152,77.575758,83.606061,71.393939,2010,Tampa Bay Region,0.021739,0.0,8.180801,27.982726,91.771899
2830,Urophycis floridana,Southern Hake,1.454545,86.363636,89.636364,83.818182,2011,Tampa Bay Region,0.014440,0.0,8.420949,27.163580,96.664062
2831,Urophycis floridana,Southern Hake,1.000000,47.500000,47.500000,47.500000,2012,Tampa Bay Region,0.000000,0.0,8.333787,26.388476,99.303274


In [84]:
aggregate_data.corr(method='spearman', numeric_only=True)

,SAMPLE_SIZE,AVG_SIZE,MAX_SIZE,MIN_SIZE,YEAR,NH3_N_MGL,NO3_MGL,PH,SALINITY,WATER_TEMP
SAMPLE_SIZE,1.000000,-0.190921,-0.135223,-0.238310,-0.046555,-0.015194,NaN,-0.007526,0.008219,-0.029172
AVG_SIZE,-0.190921,1.000000,0.996277,0.997000,0.006981,-0.009746,NaN,-0.016810,-0.009318,0.002904
MAX_SIZE,-0.135223,0.996277,1.000000,0.987476,0.003084,-0.011935,NaN,-0.019735,-0.010839,-0.001032
MIN_SIZE,-0.238310,0.997000,0.987476,1.000000,0.009918,-0.007596,NaN,-0.014269,-0.007912,0.005605
YEAR,-0.046555,0.006981,0.003084,0.009918,1.000000,0.358252,NaN,0.419653,-0.296550,0.749402
NH3_N_MGL,-0.015194,-0.009746,-0.011935,-0.007596,0.358252,1.000000,NaN,0.749462,0.167470,0.591572
NO3_MGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PH,-0.007526,-0.016810,-0.019735,-0.014269,0.419653,0.749462,NaN,1.000000,0.248601,0.801126
SALINITY,0.008219,-0.009318,-0.010839,-0.007912,-0.296550,0.167470,NaN,0.248601,1.000000,0.092218
WATER_TEMP,-0.029172,0.002904,-0.001032,0.005605,0.749402,0.591572,NaN,0.801126,0.092218,1.000000


In [86]:
dock_aggregate_data = (
    dock_revenue
    .merge(filtered_fish, how='inner', on='YEAR')
)
dock_aggregate_data = (
    dock_aggregate_data
    .merge(region_revenue, how='inner', on='YEAR')
)
dock_aggregate_data

,YEAR,CARGO_Dockage,CARGO_Wharfage,CRUISE_Dockage,CRUISE_Wharfage,OTHER_Dockage,OTHER_Wharfage,SCIENTIFIC_NAME,COMMON_NAME,SAMPLE_SIZE,...,MIN_SIZE,REGION_NAME,DRY,GENERAL,LIQUID,PASSENGER_COUNT,REVENUE,LAND_LEASE,NET_INVESTMENT,PORT_USAGE_FEE
0,2016,5813,9791,1032.0,6371.0,659,8706,Acanthostracion quadricornis,Scrawled Cowfish,3.19863,...,89.089041,Tampa Bay Region,4458,1033,8650,813800.0,9879538.0,11752,581841,36793
1,2016,5813,9791,1032.0,6371.0,659,8706,Acanthostracion quadricornis,Scrawled Cowfish,3.19863,...,89.089041,Tampa Bay Region,4458,1033,8650,813800.0,9879538.0,26291,730710,60856
2,2016,5813,9791,1032.0,6371.0,659,8706,Acanthostracion quadricornis,Scrawled Cowfish,3.19863,...,89.089041,Tampa Bay Region,4458,1033,8650,813800.0,9879538.0,18040,641903,46206
3,2016,5813,9791,1032.0,6371.0,659,8706,Acanthostracion quadricornis,Scrawled Cowfish,3.19863,...,89.089041,Tampa Bay Region,4458,1033,8650,813800.0,9879538.0,18090,645586,37480
4,2016,5813,9791,1032.0,6371.0,659,8706,Acanthostracion quadricornis,Scrawled Cowfish,3.19863,...,89.089041,Tampa Bay Region,4458,1033,8650,813800.0,9879538.0,23288,661548,35037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130195,2023,11446,14266,1144.0,10791.0,2279,10853,Urophycis floridana,Southern Hake,1.00000,...,34.000000,Tampa Bay Region,5613,1821,9891,1149289.0,13918893.0,23605,680034,46100
130196,2023,11446,14266,1144.0,10791.0,2279,10853,Urophycis floridana,Southern Hake,1.00000,...,34.000000,Tampa Bay Region,5613,1821,9891,1149289.0,13918893.0,25662,705783,59492
130197,2023,11446,14266,1144.0,10791.0,2279,10853,Urophycis floridana,Southern Hake,1.00000,...,34.000000,Tampa Bay Region,5613,1821,9891,1149289.0,13918893.0,13058,594645,39916
130198,2023,11446,14266,1144.0,10791.0,2279,10853,Urophycis floridana,Southern Hake,1.00000,...,34.000000,Tampa Bay Region,5613,1821,9891,1149289.0,13918893.0,31044,790607,70828


In [87]:
dock_aggregate_data.corr(method='spearman', numeric_only=True)

,YEAR,CARGO_Dockage,CARGO_Wharfage,CRUISE_Dockage,CRUISE_Wharfage,OTHER_Dockage,OTHER_Wharfage,SAMPLE_SIZE,AVG_SIZE,MAX_SIZE,MIN_SIZE,DRY,GENERAL,LIQUID,PASSENGER_COUNT,REVENUE,LAND_LEASE,NET_INVESTMENT,PORT_USAGE_FEE
YEAR,1.000000,0.925647,0.904273,-0.311982,0.094664,0.848772,0.828204,-0.020569,-0.015538,-0.017063,-0.014956,0.707964,0.777306,0.597624,0.000000,0.000000,0.000000,0.000000,0.000000
CARGO_Dockage,0.925647,1.000000,0.978626,-0.203382,0.169168,0.699696,0.679128,-0.031733,-0.014030,-0.017258,-0.012036,0.708626,0.902026,0.644329,0.000000,0.000000,0.000000,0.000000,0.000000
CARGO_Wharfage,0.904273,0.978626,1.000000,-0.203382,0.169168,0.655586,0.614052,-0.033848,-0.010996,-0.014265,-0.008993,0.795758,0.859686,0.665704,0.000000,0.000000,0.000000,0.000000,0.000000
CRUISE_Dockage,-0.311982,-0.203382,-0.203382,1.000000,0.795422,-0.311880,-0.408400,0.009179,0.015864,0.017844,0.015376,-0.213553,-0.263730,0.323771,0.000000,0.000000,0.000000,0.000000,0.000000
CRUISE_Wharfage,0.094664,0.169168,0.169168,0.795422,1.000000,0.130739,-0.063677,-0.006443,0.018858,0.019649,0.018698,0.056879,-0.024220,0.632505,0.000000,0.000000,0.000000,0.000000,0.000000
OTHER_Dockage,0.848772,0.699696,0.655586,-0.311880,0.130739,1.000000,0.979432,-0.002624,-0.012437,-0.012227,-0.012602,0.365936,0.474339,0.671306,0.000000,0.000000,0.000000,0.000000,0.000000
OTHER_Wharfage,0.828204,0.679128,0.614052,-0.408400,-0.063677,0.979432,1.000000,0.000068,-0.016448,-0.016207,-0.016517,0.323739,0.494907,0.585983,0.000000,0.000000,0.000000,0.000000,0.000000
SAMPLE_SIZE,-0.020569,-0.031733,-0.033848,0.009179,-0.006443,-0.002624,0.000068,1.000000,-0.121679,-0.066818,-0.169001,-0.026324,-0.035069,-0.009724,0.000000,0.000000,0.000000,0.000000,0.000000
AVG_SIZE,-0.015538,-0.014030,-0.010996,0.015864,0.018858,-0.012437,-0.016448,-0.121679,1.000000,0.996327,0.997231,-0.013538,-0.010682,-0.005249,0.000000,0.000000,0.000000,0.000000,0.000000
MAX_SIZE,-0.017063,-0.017258,-0.014265,0.017844,0.019649,-0.012227,-0.016207,-0.066818,0.996327,1.000000,0.987980,-0.015970,-0.014268,-0.005824,0.000000,0.000000,0.000000,0.000000,0.000000


In [88]:
grass_aggregate_data = (
    filtered_fish
    .merge(yearly_acrage, how='inner', on='YEAR')
)

In [89]:
grass_aggregate_data.corr(method='spearman', numeric_only=True)

,SAMPLE_SIZE,AVG_SIZE,MAX_SIZE,MIN_SIZE,YEAR,ACRES
SAMPLE_SIZE,1.000000,-0.143603,-0.087612,-0.191956,-0.034539,-0.032884
AVG_SIZE,-0.143603,1.000000,0.996331,0.996885,-0.005288,0.009910
MAX_SIZE,-0.087612,0.996331,1.000000,0.987342,-0.009987,0.005862
MIN_SIZE,-0.191956,0.996885,0.987342,1.000000,-0.001528,0.013080
YEAR,-0.034539,-0.005288,-0.009987,-0.001528,1.000000,0.797144
ACRES,-0.032884,0.009910,0.005862,0.013080,0.797144,1.000000


In [100]:
(filtered_fish['SCIENTIFIC_NAME'] + ' - ' + filtered_fish['COMMON_NAME']).drop_duplicates().tolist()

['Acanthostracion quadricornis - Scrawled Cowfish',
 'Achirus lineatus - Lined Sole',
 'Aetobatus narinari - Spotted Eagle Ray',
 'Albula vulpes - Bonefish',
 'Alburnops petersoni - Coastal Shiner',
 'Aluterus schoepfii - Orange Filefish',
 'Ameiurus catus - White Catfish',
 'Anarchopterus criniger - Fringed Pipefish',
 'Anchoa cubana - Cuban Anchovy',
 'Anchoa hepsetus - Striped Anchovy',
 'Anchoa mitchilli - Bay Anchovy',
 'Anchoa spp. - Anchovies',
 'Ancylopsetta quadrocellata - Ocellated Flounder',
 'Archosargus probatocephalus - Sheepshead',
 'Ariopsis felis - Hardhead Catfish',
 'Astroscopus y-graecum - Southern Stargazer',
 'Bagre marinus - Gafftopsail Catfish',
 'Bairdiella chrysoura - Silver Perch',
 'Bathygobius soporator - Frillfin Goby',
 'Belonesox belizanus - Pike Killifish',
 'Brevoortia spp. - Menhadens',
 'Calamus arctifrons - Grass Porgy',
 'Calamus penna - Sheepshead Porgy',
 'Calamus spp. - porgies',
 'Callinectes ornatus - Shelligs',
 'Callinectes sapidus - Blue Cr

In [96]:
filtered_fish[['SCIENTIFIC_NAME', 'COMMON_NAME']].drop_duplicates().iloc[0]['SCIENTIFIC_NAME']

'Acanthostracion quadricornis'